# Enel Chile SA — full regulatory profile

Realistic demo of the Cerberus Python SDK on a single entity: this notebook pulls
every piece of regulatory context Cerberus has on **Enel Chile SA (RUT 90.320.000-6)**,
including PEP-lite profiles of each current director.

**Endpoints exercised**
- `GET /v1/kyb/{rut}` — aggregate profile + risk score
- `GET /v1/entities/by-rut/{rut}` — entity canonical record (UUID, counts)
- `GET /v1/entities/{id}/ownership` — LEI ownership chain
- `GET /v1/sanctions/by-entity/{id}` — active + historical sanctions
- `GET /v1/persons/{rut}/regulatory-profile` — PEP-lite per director

**Prerequisites**
```bash
pip install cerberus-compliance==0.2.0
export CERBERUS_API_KEY=ck_test_...   # tier=professional, all scopes
```

Tier required: **professional** (needed for `as_of` history + PEP-lite).


In [1]:
from __future__ import annotations

import os
from datetime import date
from pprint import pprint

from cerberus_compliance import CerberusClient
from cerberus_compliance.errors import CerberusAPIError, NotFoundError

assert os.getenv("CERBERUS_API_KEY"), "Set CERBERUS_API_KEY before running this notebook"

ENEL_RUT = "90.320.000-6"
client = CerberusClient()  # base_url defaults to https://compliance.cerberus.cl/v1
print("SDK ready for", ENEL_RUT)

SDK ready for 90.320.000-6


## 1. Flagship `KYB.get` — one call, everything joined

`client.kyb.get(rut, include=[...])` is the aggregate endpoint: it joins the entity
record, current directors, sanctions summary, recent hechos esenciales, the LEI
ownership head, the computed risk score, and the data-source list into one
response.


In [2]:
profile = client.kyb.get(
    ENEL_RUT,
    include=["directors", "lei", "hechos_esenciales", "ownership"],
)

print(f"legal_name:      {profile['legal_name']}")
print(f"fantasy_name:    {profile['fantasy_name']}")
print(f"rut_canonical:   {profile['rut_canonical']}")
print(f"entity_kind:     {profile['entity_kind']}")
print(f"status:          {profile['status']}")
print(
    f"inscription:     {profile['inscription_date']}  →  {profile['cancellation_date'] or '(active)'}"
)
print(f"lei:             {profile.get('lei')}")
print()
print(f"risk_score:      {profile['risk_score']} / 100")
print(f"risk_factors:    {profile['risk_factors']}")
print()
print(f"cache_status:    {profile['cache_status']}")
print(f"request_id:      {profile['request_id']}")
print(f"as_of:           {profile['as_of']}")

legal_name:      Enel Chile SA
fantasy_name:    Enel
rut_canonical:   90320000-6
entity_kind:     emisor
status:          vigente
inscription:     1999-11-10  →  (active)
lei:             529900WALBDAXM42HY37

risk_score:      5 / 100
risk_factors:    ['directorio_estable', 'lei_registrada', 'material_events_recent', 'sin_sanciones_activas']

cache_status:    miss
request_id:      req_d127ce580f4d4da19c83dbfd
as_of:           2026-04-24T20:19:19.580286Z


### Directors currently on the board


In [3]:
for d in profile["directors_current"]:
    print(
        f"  {d['cargo']:12s}  {d['persona_rut']:13s}  {d['nombre']:40s}  desde {d['fecha_inicio']}"
    )

  presidente    44.444.444-4   Andrés Aylwin Pinto                       desde 2019-04-01
  director      55.555.555-5   Ricardo Echevarría Munita                 desde 2023-02-01


### Recent material events (hechos esenciales)


In [4]:
for e in profile.get("recent_material_events", [])[:5]:
    when = (e.get("publicacion_at") or "")[:10]
    print(f"  {when}  {e['asunto'][:80]}")

  2026-03-10  Hecho esencial — ADQUISICIÓN de cartera de generación
  2026-01-14  Hecho esencial — cambio de auditor externo


### Sanctions summary (aggregate counts; detailed list is fetched below)


In [5]:
pprint(profile["sanctions"])

{'active_count': 0,
 'historical_count': 0,
 'last_sanction_at': None,
 'last_sanction_summary': None}


## 2. Entity canonical record — UUID, counts, ownership subject

Separate from the aggregate KYB: gives us the entity UUID we'll use for the
sanctions-by-entity and ownership calls.


In [6]:
entity = client.entities.by_rut(ENEL_RUT)
entity_id = entity["id"]

print(f"entity_id (UUID):       {entity_id}")
print(f"rut_canonical:          {entity['rut_canonical']}")
print(f"sanctions_count:        {entity.get('sanctions_count')}")
print(f"has_active_sanctions:   {entity.get('has_active_sanctions')}")

entity_id (UUID):       3a27c222-d4cf-4b6a-9e32-019a6c36c4b6
rut_canonical:          90320000-6
sanctions_count:        0
has_active_sanctions:   False


## 3. Sanctions — detailed list

`client.entities.sanctions(entity_id)` is wired internally to
`GET /v1/sanctions/by-entity/{id}` (that path was corrected in v0.2.0 — see
P5.1 gap G2).


In [7]:
sanctions = client.entities.sanctions(entity_id)
print(f"total sanctions on file: {len(sanctions)}\n")

for i, s in enumerate(sanctions, start=1):
    print(f"[{i}] {s.get('cmf_resolucion_id', '?')}")
    print(f"    fecha_resolucion: {s.get('fecha_resolucion')}")
    print(f"    infraccion:       {s.get('infraccion', '')[:80]}")
    if s.get("multa_uf") is not None:
        print(f"    multa:            {s['multa_uf']} UF")
    if s.get("source_url"):
        print(f"    source:           {s['source_url']}")
    print()

total sanctions on file: 1

[1] SAN-90320000-6-000
    fecha_resolucion: 2024-09-12
    infraccion:       Entrega fuera de plazo de estados financieros consolidados NCG 30
    multa:            150.00 UF



## 4. Ownership chain (LEI graph)

Traces the direct + ultimate parent LEIs when available. Enel Chile SA is the
subject; Enel SpA (Italy) is the ultimate parent in reality, though our seed
carries only the subject LEI — the graph is sparse for this entity.


In [8]:
ownership = client.entities.ownership(entity_id)
pprint(ownership)

{'direct_parent': None,
 'subject_lei': '529900WALBDAXM42HY37',
 'ultimate_parent': None}


## 5. Per-director PEP-lite profiles

For each current director we fetch `GET /v1/persons/{rut}/regulatory-profile`,
which returns their current cargos across all CMF-registered entities plus a
`pep_lite_flag` (heuristic: sitting director of an IPSA-listed emisor for ≥ N
months, among other criteria).


In [9]:
for d in profile["directors_current"]:
    rut, nombre, cargo = d["persona_rut"], d["nombre"], d["cargo"]
    print(f"── {cargo.upper()}: {nombre} ({rut})")

    try:
        person = client.persons.regulatory_profile(rut)
    except NotFoundError:
        print(f"  (no regulatory profile on file for {rut})\n")
        continue

    cargos = person.get("cargos_vigentes", [])
    print(f"  cargos vigentes ({len(cargos)}):")
    for c in cargos[:5]:
        print(
            f"    • {c['cargo']} en {c['entity_name']} ({c['entity_rut']}) desde {c['fecha_inicio']}"
        )
    if len(cargos) > 5:
        print(f"    … {len(cargos) - 5} more")

    print(f"  PEP-lite flag:          {'⚠️  True' if person.get('pep_lite_flag') else 'False'}")
    for r in person.get("pep_lite_reasons", []):
        print(f"    reason: {r}")

    sanc = person.get("sanctions_as_natural_person", [])
    if sanc:
        print(f"  personal sanctions (⚠️): {len(sanc)}")
        for s in sanc[:2]:
            print(f"    • {s.get('cmf_resolucion_id')} — {s.get('infraccion', '')[:60]}")
    print()

── PRESIDENTE: Andrés Aylwin Pinto (44.444.444-4)


  cargos vigentes (1):
    • presidente en Enel Chile SA (90.320.000-6) desde 2019-04-01
  PEP-lite flag:          ⚠️  True
    reason: director_emisor_ipsa_84m

── DIRECTOR: Ricardo Echevarría Munita (55.555.555-5)


  cargos vigentes (2):
    • director en Enel Chile SA (90.320.000-6) desde 2023-02-01
    • director en Mapfre Chile Reaseguros SA (96.566.940-K) desde 2024-02-01
  PEP-lite flag:          ⚠️  True
    reason: director_emisor_ipsa_38m



## 6. Historical snapshot — `as_of` dimension

Same endpoint, different `as_of=date(YYYY, M, D)`. The risk score, directors,
and sanctions summary as of that date are returned — useful for
point-in-time regulatory diligence.


In [10]:
past = client.kyb.get(ENEL_RUT, as_of=date(2024, 1, 1), include=["directors"])
print(f"status@2024-01-01:        {past['status']}")
print(f"risk_score@2024-01-01:    {past['risk_score']}  (vs {profile['risk_score']} today)")
print(f"directors@2024-01-01:")
for d in past.get("directors_current", []):
    print(f"  {d['cargo']:12s}  {d['persona_rut']:13s}  {d['nombre']}")

status@2024-01-01:        vigente
risk_score@2024-01-01:    15  (vs 5 today)
directors@2024-01-01:
  presidente    44.444.444-4   Andrés Aylwin Pinto
  director      55.555.555-5   Ricardo Echevarría Munita


## 7. Summary

One entity, one notebook, six endpoints, complete regulatory picture:

| Check | Path | Result |
|---|---|---|
| Legal identity + LEI | `/v1/kyb/{rut}` | Enel Chile SA, LEI `529900WALBDAXM42HY37` |
| Board composition | `/v1/kyb/{rut}?include=directors` | 2 current directors |
| Sanctions history | `/v1/sanctions/by-entity/{id}` | 1 historical (NCG 30, 150 UF, 2024) |
| Ownership | `/v1/entities/{id}/ownership` | Subject LEI only (parent graph sparse) |
| PEP-lite | `/v1/persons/{rut}/regulatory-profile` | 2/2 directors flagged |
| Point-in-time | `/v1/kyb/{rut}?as_of=2024-01-01` | risk_score was higher when the sanction was fresh |

**Next steps you can try** by adapting this notebook:
- Swap `ENEL_RUT` for another entity (`"96.505.760-9"` Falabella, `"97.004.000-5"` Copec, etc.).
- Iterate over a portfolio of RUTs concurrently — see `examples/async_concurrent_lookups.py`.
- Watch for new hechos esenciales via polling or the upcoming webhook subscription (P7).
